In [1]:
# Discovery Notebook — QEC Data Pipeline
## Steps 1–4: Setup, Inventory, Validation, Structure

In [2]:
import zipfile
import io
import yaml
import pandas as pd
from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client, bronze_inventory
from quantum_lake_student.formats import iter_b8_records, parse_01_records

settings = Settings.from_environment()
client = minio_client(settings)

In [3]:
def read_zip_member(bronze_key, member_name):
    response = client.get_object(settings.s3_bucket, bronze_key)
    data = response.read()
    response.close()
    response.release_conn()
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        return zf.read(member_name)

def list_zip_members(bronze_key):
    response = client.get_object(settings.s3_bucket, bronze_key)
    data = response.read()
    response.close()
    response.release_conn()
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        return [(name, zf.getinfo(name).file_size) for name in zf.namelist()]

In [4]:
google_key = "bronze/source=google_qec/google-surface-code-curated.zip"
syndromes_key = "bronze/source=qec_syndromes/syndromes_dataset.zip"
qasmbench_key = "bronze/source=qasmbench/qasmbench-qec.zip"

In [5]:
## Step 2: Inventory — what's inside each Bronze archive

In [6]:
for key in [google_key, syndromes_key, qasmbench_key]:
    print(f"\n=== {key} ===")
    for name, size in list_zip_members(key):
        print(f"  {name}  ({size:,} bytes)")


=== bronze/source=google_qec/google-surface-code-curated.zip ===
  README.txt  (15,656 bytes)
  surface_code_bX_d3_r25_center_3_5/circuit_detector_error_model.dem  (172,277 bytes)
  surface_code_bX_d3_r25_center_3_5/circuit_ideal.stim  (18,773 bytes)
  surface_code_bX_d3_r25_center_3_5/circuit_noisy.stim  (124,790 bytes)
  surface_code_bX_d3_r25_center_3_5/detection_events.b8  (1,250,000 bytes)
  surface_code_bX_d3_r25_center_3_5/layout.svg  (17,685 bytes)
  surface_code_bX_d3_r25_center_3_5/measurements.b8  (1,350,000 bytes)
  surface_code_bX_d3_r25_center_3_5/obs_flips_actual.01  (100,000 bytes)
  surface_code_bX_d3_r25_center_3_5/obs_flips_predicted_by_belief_matching.01  (100,000 bytes)
  surface_code_bX_d3_r25_center_3_5/obs_flips_predicted_by_correlated_matching.01  (100,000 bytes)
  surface_code_bX_d3_r25_center_3_5/obs_flips_predicted_by_pymatching.01  (100,000 bytes)
  surface_code_bX_d3_r25_center_3_5/obs_flips_predicted_by_tensor_network_contraction.01  (100,000 bytes)
  su

In [7]:
## Step 3: Validate one sample from each source

In [8]:
exp_dir = "surface_code_bX_d3_r25_center_3_5"

props = yaml.safe_load(read_zip_member(google_key, f"{exp_dir}/properties.yml"))
print("Properties:", props)

meas_bytes = read_zip_member(google_key, f"{exp_dir}/measurements.b8")
first_measurement = next(iter_b8_records(meas_bytes, bits_per_record=props["circuit_measurements"]))
print("\nFirst measurement record (len={}):".format(len(first_measurement)), first_measurement)

det_bytes = read_zip_member(google_key, f"{exp_dir}/detection_events.b8")
first_detection = next(iter_b8_records(det_bytes, bits_per_record=props["circuit_detectors"]))
print("\nFirst detection record (len={}):".format(len(first_detection)), first_detection)

obs_bytes = read_zip_member(google_key, f"{exp_dir}/obs_flips_actual.01")
obs_flips = parse_01_records(obs_bytes)
print("\nFirst 10 obs_flips_actual values:", obs_flips[:10])
print("Total shots:", len(obs_flips))

Properties: {'type': 'surface_code_memory_experiment', 'basis': 'X', 'rounds': 25, 'distance': 3, 'data_qubits': 9, 'measure_qubits': 8, 'shots': 50000, 'center_data_qubit_row': 3, 'center_data_qubit_col': 5, 'circuit_measurements': 209, 'circuit_sweep_bits': 9, 'circuit_detectors': 200, 'circuit_observables': 1, 'circuit_qubits': 17}

First measurement record (len=209): (1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1

In [9]:
syn_bytes = read_zip_member(syndromes_key, "d-3_pfr-0.000010_nb-10M.csv")
syn_df = pd.read_csv(io.BytesIO(syn_bytes))
print(syn_df.shape)
print(syn_df.dtypes)
syn_df.head(3)

(68, 3)
labels        int64
syndromes    object
quantity      int64
dtype: object


,labels,syndromes,quantity
0,0,"((0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 0, 0), (0,...",9987291
1,0,"((0, 0, 1, 0), (0, 0, 1, 0), (0, 0, 0, 0), (0,...",486
2,1,"((0, 0, 1, 0), (0, 0, 0, 0), (0, 0, 0, 0), (0,...",476


In [10]:
for family in ["error_correctiond3_n5", "qec_en_n5", "qec_sm_n5"]:
    print(f"\n=== {family}.qasm ===")
    print(read_zip_member(qasmbench_key, f"small/{family}/{family}.qasm").decode())


=== error_correctiond3_n5.qasm ===
// Error correction: distance-three 5-qubit code, from the paper "Benchmarking gate-based quantum computers" by K. Michielsen et al.

OPENQASM 2.0;
include "qelib1.inc";

qreg q[5];
creg c[5];

h q[0];
h q[1];
id q[2];
h q[3];
h q[4];
cx q[1],q[2];
h q[1];
h q[2];
cx q[1],q[2];
h q[1];
h q[2];
cx q[1],q[2];
cx q[4],q[2];
cx q[1],q[2];
h q[1];
h q[2];
cx q[1],q[2];
h q[1];
h q[2];
cx q[1],q[2];
sdg q[4];
cx q[4],q[2];
h q[2];
cx q[4],q[2];
h q[2];
cx q[0],q[2];
h q[0];
h q[2];
cx q[0],q[2];
h q[0];
h q[2];
cx q[0],q[2];
cx q[3],q[2];
cx q[0],q[2];
h q[0];
h q[2];
cx q[0],q[2];
h q[0];
h q[2];
cx q[0],q[2];
cx q[1],q[2];
h q[1];
h q[2];
cx q[1],q[2];
h q[1];
h q[2];
cx q[1],q[2];
cx q[3],q[2];
cx q[1],q[2];
h q[1];
h q[2];
cx q[1],q[2];
h q[1];
h q[2];
cx q[1],q[2];
cx q[3],q[2];
cx q[0],q[2];
h q[3];
h q[4];
cx q[3],q[2];
h q[2];
h q[3];
cx q[3],q[2];
h q[2];
h q[3];
cx q[3],q[2];
cx q[0],q[2];
cx q[3],q[2];
h q[2];
h q[3];
cx q[3],q[2];
h q[2];
h q[3

In [11]:
## Step 4: Structure summary — sizes, counts, columns/dtypes, candidate IDs

In [12]:
for family in ["error_correctiond3_n5", "qec_en_n5", "qec_sm_n5"]:
    print(f"\n=== {family}/README.md ===")
    print(read_zip_member(qasmbench_key, f"small/{family}/README.md").decode())


=== error_correctiond3_n5/README.md ===
# Application: error_correctiond3_n5
- Qubit Count : 5
- Circuit Depth : 78
- Circuit Width : 5
- Retention Lifespan : 4.356708826689592
- Gate Density : 0.41794871794871796
- Dual Gate Count : 49
- Measurement Density : 1.1932293478247384
- Size Factor : 5.093750200806762
- Gate Count : 114
- Entanglement Variance : 1.390583966187302
- Communication Supermarq : 0.5
- Measurement Supermarq : 0.0
- Depth Supermarq : 0.9591836734693877
- Entanglement Supermarq : 0.4298245614035088
- Parallelism Supermarq : 0.3157894736842105
- Liveness Supermarq : 0.4307692307692308


=== qec_en_n5/README.md ===
# Application: qec_en_n5
- Qubit Count : 5
- Circuit Depth : 18
- Circuit Width : 5
- Retention Lifespan : 2.8903717578961645
- Gate Density : 0.3888888888888889
- Dual Gate Count : 10
- Measurement Density : 0.899961934066053
- Size Factor : 4.174387269895637
- Gate Count : 25
- Entanglement Variance : 1.0396994062531653
- Communication Supermarq : 0.4
- 

In [13]:
print("qec_syndromes:")
print(f"  columns: {list(syn_df.columns)}")
print(f"  dtypes: {dict(syn_df.dtypes)}")
print(f"  candidate keys: (labels, syndromes) per row within a file; filename encodes physical_fault_rate")

print("\ngoogle_qec:")
print(f"  candidate experiment_id: directory name, e.g. '{exp_dir}'")
print(f"  candidate shot_index: row position within measurements.b8 / detection_events.b8 / obs_flips_*.01")
print(f"  properties per experiment: {props}")

print("\nqasmbench:")
print(f"  candidate circuit_id / benchmark_name: folder name, e.g. 'qec_sm_n5'")
print(f"  variant: source vs transpiled .qasm file")

qec_syndromes:
  columns: ['labels', 'syndromes', 'quantity']
  dtypes: {'labels': dtype('int64'), 'syndromes': dtype('O'), 'quantity': dtype('int64')}
  candidate keys: (labels, syndromes) per row within a file; filename encodes physical_fault_rate

google_qec:
  candidate experiment_id: directory name, e.g. 'surface_code_bX_d3_r25_center_3_5'
  candidate shot_index: row position within measurements.b8 / detection_events.b8 / obs_flips_*.01
  properties per experiment: {'type': 'surface_code_memory_experiment', 'basis': 'X', 'rounds': 25, 'distance': 3, 'data_qubits': 9, 'measure_qubits': 8, 'shots': 50000, 'center_data_qubit_row': 3, 'center_data_qubit_col': 5, 'circuit_measurements': 209, 'circuit_sweep_bits': 9, 'circuit_detectors': 200, 'circuit_observables': 1, 'circuit_qubits': 17}

qasmbench:
  candidate circuit_id / benchmark_name: folder name, e.g. 'qec_sm_n5'
  variant: source vs transpiled .qasm file
